# Lab 0: Download DAS Event Data from HuggingFace

This notebook downloads DAS waveform data for the 25 events used in the dasfm workshop.

**Data source:** [AI4EPS/quakeflow_das](https://huggingface.co/datasets/AI4EPS/quakeflow_das/tree/main/polarity/data)  
**Target directory:** `../Inputs/proj_25events/input/das_raw/H5/`

In [ ]:
import os
import shutil
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files

## 1. Configuration

In [ ]:
REPO_ID = "AI4EPS/quakeflow_das"
SUBSET = "polarity"  # subset name on HuggingFace

# paths relative to the repo root
PROJECT_DIR = Path("../Inputs/proj_25events")
CATALOG_PATH = PROJECT_DIR / "input" / "catalog_25events.csv"
OUTPUT_DIR = PROJECT_DIR / "input" / "das_raw" / "H5"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Catalog:    {CATALOG_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

## 2. Read event catalog

In [ ]:
catalog = pd.read_csv(CATALOG_PATH)
event_ids = catalog["event_id"].tolist()
print(f"{len(event_ids)} events in catalog")

# check which events actually exist on HuggingFace
hf_files = list_repo_files(REPO_ID, repo_type="dataset")
hf_event_ids = {
    f.split("/")[-1].replace(".h5", "")
    for f in hf_files
    if f.startswith(f"{SUBSET}/data/") and f.endswith(".h5")
}

available = [eid for eid in event_ids if eid in hf_event_ids]
missing = [eid for eid in event_ids if eid not in hf_event_ids]

if missing:
    print(f"\n{len(missing)} events NOT available on HuggingFace (will be skipped):")
    for eid in missing:
        print(f"  {eid}")

print(f"\n{len(available)} events available for download")

## 3. Download events from HuggingFace

Files are cached by `huggingface_hub` after the first download. Re-running this cell will skip already-downloaded files.

In [ ]:
downloaded = []
skipped = []
failed = []

for i, event_id in enumerate(available):
    local_path = OUTPUT_DIR / f"{event_id}.h5"
    repo_path = f"{SUBSET}/data/{event_id}.h5"

    if local_path.exists():
        skipped.append(event_id)
        print(f"[{i+1:3d}/{len(available)}] {event_id} — already exists, skipping")
        continue

    try:
        cached_path = hf_hub_download(
            REPO_ID,
            repo_path,
            repo_type="dataset",
        )
        # copy from HF cache to our project directory
        if not local_path.exists():
            shutil.copy2(cached_path, local_path)
        downloaded.append(event_id)
        print(f"[{i+1:3d}/{len(available)}] {event_id} — downloaded")
    except Exception as e:
        failed.append((event_id, str(e)))
        print(f"[{i+1:3d}/{len(available)}] {event_id} — FAILED: {e}")

print(f"\nDone: {len(downloaded)} downloaded, {len(skipped)} already existed, {len(failed)} failed")
if missing:
    print(f"      {len(missing)} unavailable on HuggingFace (see above)")

In [ ]:
if failed:
    print("Failed events:")
    for event_id, err in failed:
        print(f"  {event_id}: {err}")

## 4. Verify downloaded files

In [ ]:
print(f"{'event_id':<20} {'shape':>18} {'mag':>5} {'dt_s':>6} {'dx_m':>8}")
print("-" * 60)

for event_id in available:
    fpath = OUTPUT_DIR / f"{event_id}.h5"
    if not fpath.exists():
        print(f"{event_id:<20} {'MISSING':>18}")
        continue
    with h5py.File(fpath, "r") as f:
        data = f["data"]
        shape = data.shape
        mag = data.attrs.get("magnitude", -999)
        dt = data.attrs.get("dt_s", -999)
        dx = data.attrs.get("dx_m", -999)
    print(f"{event_id:<20} {str(shape):>18} {mag:5.1f} {dt:6.3f} {dx:8.2f}")

In [ ]:
# quick look at one event
sample_id = available[0]
with h5py.File(OUTPUT_DIR / f"{sample_id}.h5", "r") as f:
    data = f["data"][:]
    attrs = dict(f["data"].attrs)

print(f"Event: {sample_id}")
print(f"Shape: {data.shape}  (n_channels, n_samples)")
print(f"Duration: {data.shape[1] * attrs['dt_s']:.1f} s")
print(f"Attributes:")
for k, v in attrs.items():
    print(f"  {k}: {v}")